# Natural Language Processing
![](https://i.imgur.com/qkg2E2D.png)

## Assignment 002 - NER Tagger

> Notebook by:
> - NLP Course Staff
## Revision History

| Version | Date       | User        | Content / Changes                                                   |
|---------|------------|-------------|---------------------------------------------------------------------|
| 0.1.000 | 2026        | course staff | Updated submission protocol (direct from Colab) |

## Overview
In this assignment, you will build a complete training and testing pipeline for a neural sequential tagger for named entities using LSTM.

## Dataset
You will work with the ReCoNLL 2003 dataset, a corrected version of the [CoNLL 2003 dataset](https://www.clips.uantwerpen.be/conll2003/ner/):

**Click on the links below to download the data files.**
- [Train data](https://drive.google.com/file/d/1CqEGoLPVKau3gvVrdG6ORyfOEr1FSZGf/view?usp=sharing)

- [Dev data](https://drive.google.com/file/d/1rdUida-j3OXcwftITBlgOh8nURhAYUDw/view?usp=sharing)

- [Test data](https://drive.google.com/file/d/137Ht40OfflcsE6BIYshHbT5b2iIJVaDx/view?usp=sharing)

As you will see, the annotated texts are labeled according to the `IOB` annotation scheme (more on this below), for 3 entity types: Person, Organization, Location.

## Your Implementation

This notebook **is** the assignment template. To work on it, open it in Colab using the badge below and then **File → Save a copy in Drive** to create your own editable copy.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1KGkObwUn5QQm_v0nB0nAUlB4YrwThuzl#scrollTo=Z-fCqGh9ybgm)

Work through the cells in order — each section's instructions are written above the corresponding code cell.

Good Luck 🤗


<!-- ## NER schemes:  

> `IO`: is the simplest scheme that can be applied to this task. In this scheme, each token from the dataset is assigned one of two tags: an inside tag (`I`) and an outside tag (`O`). The `I` tag is for named entities, whereas the `O` tag is for normal words. This scheme has a limitation, as it cannot correctly encode consecutive entities of the same type.

> `IOB`: This scheme is also referred to in the literature as BIO and has been adopted by the Conference on Computational Natural Language Learning (CoNLL) [1]. It assigns a tag to each word in the text, determining whether it is the beginning (`B`) of a known named entity, inside (`I`) it, or outside (`O`) of any known named entities.

> `IOE`: This scheme works nearly identically to `IOB`, but it indicates the end of the entity (`E` tag) instead of its beginning.

> `IOBES`: An alternative to the IOB scheme is `IOBES`, which increases the amount of information related to the boundaries of named entities. In addition to tagging words at the beginning (`B`), inside (`I`), end (`E`), and outside (`O`) of a named entity. It also labels single-token entities with the tag `S`.

> `BI`: This scheme tags entities in a similar method to `IOB`. Additionally, it labels the beginning of non-entity words with the tag B-O and the rest as I-O.

> `IE`: This scheme works exactly like `IOE` with the distinction that it labels the end of non-entity words with the tag `E-O` and the rest as `I-O`.

> `BIES`: This scheme encodes the entities similar to `IOBES`. In addition, it also encodes the non-entity words using the same method. It uses `B-O` to tag the beginning of non-entity words, `I-O` to tag the inside of non-entity words, and `S-O` for single non-entity tokens that exist between two entities. -->


## NER Schemes

### IO
- **Description**: The simplest scheme for named entity recognition (NER).
- **Tags**:
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
- **Limitation**: Cannot correctly encode consecutive entities of the same type.

### IOB (BIO)
- **Description**: Adopted by the Conference on Computational Natural Language Learning (CoNLL).
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
- **Advantage**: Can encode the boundaries of consecutive entities.

### IOE
- **Description**: Similar to IOB, but indicates the end of an entity.
- **Tags**:
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
- **Advantage**: Focuses on the end boundary of entities.

### IOBES
- **Description**: An extension of IOB with additional boundary information.
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
  - `S`: Single-token named entity.
- **Advantage**: Provides more detailed boundary information for named entities.

### BI
- **Description**: Tags entities similarly to IOB and labels the beginning of non-entity words.
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `B-O`: Beginning of a non-entity word.
  - `I-O`: Inside a non-entity word.
- **Advantage**: Distinguishes the beginning of non-entity sequences.

### IE
- **Description**: Similar to IOE but for non-entity words.
- **Tags**:
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
  - `E-O`: End of a non-entity word.
  - `I-O`: Inside a non-entity word.
- **Advantage**: Highlights the end of non-entity sequences.

### BIES
- **Description**: Encodes both entities and non-entity words using the IOBES method.
- **Tags**:
  - `B`: Beginning of a named entity.
  - `I`: Inside a named entity.
  - `O`: Outside any named entity.
  - `E`: End of a named entity.
  - `S`: Single-token named entity.
  - `B-O`: Beginning of a non-entity word.
  - `I-O`: Inside a non-entity word.
  - `S-O`: Single non-entity token.
- **Advantage**: Comprehensive encoding for both entities and non-entities.




# Set up

In [ ]:
# !git clone https://github.com/NLP-Reichman/assignment-2-ner-lidors.git
# # Move into the cloned repo so the `data/...` paths below resolve.
# %cd assignment-2-ner-lidors

import os
os.chdir('/Users/lidor/MLDS/Y2S2/NLP/HW2/assignment-2-ner-lidors')

In [ ]:
# Any additional needed libraries
import sys
!{sys.executable} -m pip install -q seaborn tqdm tabulate torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [ ]:
# Standard Library Imports
import os
import copy
import random
import warnings
from collections import defaultdict
from typing import Optional

# ML
import numpy as np
import scipy as sp
import pandas as pd

# Visual
import matplotlib
import seaborn as sns
from tqdm import tqdm
from tabulate import tabulate
import matplotlib.pyplot as plt
from IPython.display import display

# DL
import torch as th
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

# Metrics
from sklearn import metrics
from sklearn.metrics import accuracy_score , roc_auc_score, classification_report, confusion_matrix, precision_recall_fscore_support

try:
    from google.colab import files
    COLAB = True
except ImportError:
    COLAB = False

In [ ]:
SEED = 42
# Set the random seed for Python
random.seed(SEED)

# Set the random seed for numpy
np.random.seed(SEED)

# Set the random seed for pytorch
th.manual_seed(SEED)

# If using CUDA (for GPU operations)
th.cuda.manual_seed(SEED)

# Set up the device — this assignment expects a GPU runtime.
# In Colab: Runtime → Change runtime type → T4 GPU (or any available GPU).
# TO DO ----------------------------------------------------------------------
DEVICE = "cpu"
# TO DO ----------------------------------------------------------------------
# assert th.cuda.is_available(), "No GPU detected. In Colab: Runtime → Change runtime type → GPU."
# assert DEVICE == "cuda", "DEVICE must be set to 'cuda'."

DataType = list[tuple[list[str],list[str]]]

# Part 1 - Dataset Preparation

## Step 1: Read Data
Write a function for reading the data from a single file (of the ones that are provided above).   
- The function recieves a filepath
- The funtion encodes every sentence individually using a pair of lists, one list contains the words and one list contains the tags.
- Each list pair will be added to a general list (data), which will be returned back from the function.

Example output:
```
[
  (['At','Trent','Bridge',':'],['O','B-LOC','I-LOC','O']),
  ([...],[...]),
  ...
]
```

In [ ]:
def read_data(filepath:str) -> DataType:
  """
  Read data from a single file.
  The function recieves a filepath
  The funtion encodes every sentence using a pair of lists, one list contains the words and one list contains the tags.
  :param filepath: path to the file
  :return: data as a list of tuples
  """
  data = []
  # TO DO ----------------------------------------------------------------------
  words, tags = [], []
  with open(filepath, 'r') as f:
    for line in f:
      line = line.strip()
      if line == '':
        if words:
          data.append((words, tags))
          words, tags = [], []
      else:
        parts = line.split()
        words.append(parts[0])
        tags.append(parts[-1])
  if words:
    data.append((words, tags))
  # TO DO ----------------------------------------------------------------------
  return data

In [ ]:

train = read_data("data/connl03_train.txt")
dev = read_data("data/connl03_dev.txt")
test = read_data("data/connl03_test.txt")

## Step 2: Create Vocab

The `Vocab` class will serve as a dictionary that maps words and tags into IDs. Ensure that you include special tokens to handle out-of-vocabulary words and padding.

### Your Task
1. **Define Special Tokens**: Define special tokens such as `PAD_TOKEN` and `UNK_TOKEN` and assign them unique IDs.
2. **Initialize Dictionaries**: Populate the word and tag dictionaries based on the training set.

*Note: You may change the `Vocab` class as needed.*

In [ ]:
# Initialize ids for special tokens.
# These must be defined *before* the Vocab class is instantiated,
# since Vocab.__init__ references them.
PAD_TOKEN = 0
UNK_TOKEN = 1

class Vocab:
  def __init__(self, train: DataType):
    self.word2id = {"__unk__": UNK_TOKEN, "__pad__": PAD_TOKEN}
    self.id2word = {UNK_TOKEN: "__unk__", PAD_TOKEN: "__pad__"}
    self.n_words = 2

    self.tag2id = {}
    self.id2tag = {}
    self.n_tags = 0

    # Initialize dictionaries based on the training set
    # TO DO ----------------------------------------------------------------------
    for words, tags in train:
      for word in words:
        if word not in self.word2id:
          self.word2id[word] = self.n_words
          self.id2word[self.n_words] = word
          self.n_words += 1
      for tag in tags:
        if tag not in self.tag2id:
          self.tag2id[tag] = self.n_tags
          self.id2tag[self.n_tags] = tag
          self.n_tags += 1
    # TO DO ----------------------------------------------------------------------

  def __len__(self):
    return self.n_words

  def index_tags(self, tags: list[str]) -> list[int]:
    return [self.tag2id[t] for t in tags]

  def index_words(self, words: list[str]) -> list[int]:
    return [self.word2id[w] if w in self.word2id else UNK_TOKEN for w in words]

In [ ]:
vocab = Vocab(train)

In [ ]:
# Check sizes
print("Vocab size:", vocab.n_words)
print("Tag count:", vocab.n_tags)

# Check special tokens
print("\nSpecial tokens:")
print("PAD id:", vocab.word2id['__pad__'])   # should be 0
print("UNK id:", vocab.word2id['__unk__'])   # should be 1

# Check a known word
sample_word = train[0][0][0]  # first word of first sentence
print(f"\nFirst word in train: '{sample_word}'")
word_id = vocab.word2id[sample_word]
print(f"Its ID: {word_id}")
print(f"Reverse lookup: {vocab.id2word[word_id]}")

# Check a known tag
sample_tag = train[0][1][0]  # first tag of first sentence
print(f"\nFirst tag in train: '{sample_tag}'")
tag_id = vocab.tag2id[sample_tag]
print(f"Its ID: {tag_id}")
print(f"Reverse lookup: {vocab.id2tag[tag_id]}")

# Check OOV word maps to UNK
print("\nOOV word 'Spongebob' maps to:",
vocab.index_words(['Spongebob']))  # should be [1]

# Check index_words and index_tags round-trip
words, tags = train[0]
print("\nSample sentence:", words)
print("Word IDs:", vocab.index_words(words))
print("Tag IDs:", vocab.index_tags(tags))
print("All tags:", list(vocab.tag2id.keys()))

Vocab size: 7163
Tag count: 7

Special tokens:
PAD id: 0
UNK id: 1

First word in train: 'Portuguesa'
Its ID: 2
Reverse lookup: Portuguesa

First tag in train: 'B-ORG'
Its ID: 0
Reverse lookup: B-ORG

OOV word 'Spongebob' maps to: [1]

Sample sentence: ['Portuguesa', '2', 'Parana', '0']
Word IDs: [2, 3, 4, 5]
Tag IDs: [0, 1, 0, 1]
All tags: ['B-ORG', 'O', 'B-LOC', 'I-LOC', 'B-PER', 'I-PER', 'I-ORG']


## Step 3: Prepare Data
Write a function `prepare_data` that takes one of the [train, dev, test] and the `Vocab` instance, for converting each pair of (words, tags) to a pair of indexes. Additionally, the function should pad the sequences to the maximum length sequence **of the given split**.

Note: Vocabulary is based only on the train set.

### Your Task
1. Convert each pair of (words, tags) to a pair of indexes using the Vocab instance.
2. Pad the sequences to the maximum length of the sequences in the given split.

In [ ]:
def prepare_data(data: DataType, vocab: Vocab):
  data_sequences = []
  # TO DO ----------------------------------------------------------------------
  indexed = [(vocab.index_words(words), vocab.index_tags(tags)) for words, tags in data]
  max_len = max(len(w) for w, t in indexed)
  for word_ids, tag_ids in indexed:
    pad_len = max_len - len(word_ids)
    data_sequences.append((
      word_ids + [PAD_TOKEN] * pad_len,
      tag_ids  + [-100]      * pad_len   
    ))
  # TO DO ----------------------------------------------------------------------
  return data_sequences

In [ ]:
train_sequences = prepare_data(train, vocab)
dev_sequences = prepare_data(dev, vocab)
test_sequences = prepare_data(test, vocab)

### Your Task
Print the number of OOV in dev and test sets:

In [ ]:
def count_oov(sequences) -> int:
  oov = -1
  # TO DO ----------------------------------------------------------------------
  for word_ids, _ in sequences:
    oov += word_ids.count(UNK_TOKEN)

  # TO DO ----------------------------------------------------------------------
  return oov


In [ ]:
print("Dev OOV: ", count_oov(dev_sequences))
print("Test OOV:", count_oov(test_sequences))

Dev OOV:  637
Test OOV: 1367


In [ ]:
# Count how many words in test are not in the train vocabulary
oov_count = 0
for words, tags in test:
    for word in words:
        if word not in vocab.word2id:
            oov_count += 1

print("Manual OOV count:", oov_count)

oov_words = set()
for words, tags in test:
    for word in words:
        if word not in vocab.word2id:
            oov_words.add(word)

print("Unique OOV words:", len(oov_words))
print("Examples:", list(oov_words)[:10])

Manual OOV count: 1368
Unique OOV words: 1263
Examples: ['raged', 'assignments', 'limping', 'mortgage', 'Radulescu', 'building', 'benefit', 'GOLF', 'La', 'Wertpapier']


## Step 4: Dataloaders
Create dataloaders for each split in the dataset. They should return the samples as Tensors.

**Hint** - you can create a Dataset to support this part.

For the training set, use shuffling, and for the dev and test, not.

In [ ]:
class NERDataset(Dataset):
  def __init__(self, sequences):
    self.data = sequences

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    word_ids, tag_ids = self.data[idx]
    return th.tensor(word_ids, dtype=th.long), th.tensor(tag_ids, dtype=th.long)

def prepare_data_loader(sequences, batch_size: int, train: bool = True):
  """
  Create a dataloader from a list of sequences.
  :param sequences: list of sequences
  :param batch_size: batch size
  :param train: whether to shuffle the dataloader or not
  :return: dataloader
  """
  dataloader = None
  # TO DO ----------------------------------------------------------------------
  dataset = NERDataset(sequences)
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=train)
  # TO DO ----------------------------------------------------------------------
  return dataloader


In [ ]:
BATCH_SIZE = 16
dl_train = prepare_data_loader(train_sequences, batch_size=BATCH_SIZE)
dl_dev = prepare_data_loader(dev_sequences, batch_size=BATCH_SIZE, train=False)
dl_test = prepare_data_loader(test_sequences, batch_size=BATCH_SIZE, train=False)

<br><br><br><br><br><br>

# Part 2 - NER Model Training

## Step 1: Implement Model

Write NERNet, a PyTorch Module for labeling words with NER tags.

> `input_size`: the size of the vocabulary  
`embedding_size`: the size of the embeddings  
`hidden_size`: the LSTM hidden size  
`output_size`: the number of tags we are predicting  
`n_layers`: the number of layers we want to use in LSTM  
`directions`: could 1 or 2, indicating unidirectional or bidirectional LSTM, respectively  

<br>  

The input for your forward function is a batch of sentence tensors with shape `(batch_size, seq_len)`.

*Note: the embeddings in this section are learned from scratch. That means you do **not** need pretrained embeddings here — you will use those in **Part 4**.*

*Note: You may change the NERNet class.*

In [ ]:
class NERNet(nn.Module):
  def __init__(self, input_size: int, embedding_size: int, hidden_size: int, output_size: int, n_layers: int, directions: int):
    super(NERNet, self).__init__()
    self.embedding = nn.Embedding(input_size, embedding_size, padding_idx=PAD_TOKEN)
    self.lstm = nn.LSTM(
      embedding_size,
      hidden_size,
      num_layers=n_layers,
      bidirectional=(directions == 2),
      batch_first=True
    )
    self.fc = nn.Linear(hidden_size * directions, output_size)

  def forward(self, input_sentence):
    # input_sentence: (batch_size, seq_len)
    embedded = self.embedding(input_sentence)        # (batch_size, seq_len, embedding_size)
    lstm_out, _ = self.lstm(embedded)               # (batch_size, seq_len, hidden_size * directions)
    output = self.fc(lstm_out)                       # (batch_size, seq_len, output_size)
    return output

In [ ]:
model = NERNet(vocab.n_words, embedding_size=300, hidden_size=800, output_size=vocab.n_tags, n_layers=2, directions=1)
model.to(DEVICE)

## Step 2: Training Loop

Write a training loop, which takes a model (instance of NERNet), number of epochs to train on, and the train&dev datasets.  

The function will return the `loss` and `accuracy` durring training.  
(If you're using a different/additional metrics, return them too)

The loss is always CrossEntropyLoss and the optimizer is always Adam.
Make sure to use `tqdm` while iterating on `n_epochs`.


In [ ]:
def train_loop(model: NERNet, n_epochs: int, dataloader_train, dataloader_dev):
  optimizer = Adam(model.parameters(), lr=0.0001)
  criterion = nn.CrossEntropyLoss(ignore_index=-100)
  metrics = {'loss': {'train': [], 'dev': []}, 'accuracy': {'train': [], 'dev': []}}
  model.to(DEVICE)

  for epoch in tqdm(range(n_epochs)):
    # Train
    model.train()
    train_loss, correct, total = 0, 0, 0
    for word_ids, tag_ids in dataloader_train:
      word_ids, tag_ids = word_ids.to(DEVICE), tag_ids.to(DEVICE)
      optimizer.zero_grad()
      output = model(word_ids)
      loss = criterion(output.view(-1, output.shape[-1]), tag_ids.view(-1))
      loss.backward()
      optimizer.step()
      train_loss += loss.item()
      mask = tag_ids != -100
      correct += (output.argmax(-1)[mask] == tag_ids[mask]).sum().item()
      total   += mask.sum().item()
    metrics['loss']['train'].append(train_loss / len(dataloader_train))
    metrics['accuracy']['train'].append(correct / total)

    # Eval
    model.eval()
    dev_loss, correct, total = 0, 0, 0
    with th.no_grad():
      for word_ids, tag_ids in dataloader_dev:
        word_ids, tag_ids = word_ids.to(DEVICE), tag_ids.to(DEVICE)
        output = model(word_ids)
        dev_loss += criterion(output.view(-1, output.shape[-1]), tag_ids.view(-1)).item()
        mask = tag_ids != -100
        correct += (output.argmax(-1)[mask] == tag_ids[mask]).sum().item()
        total   += mask.sum().item()
    metrics['loss']['dev'].append(dev_loss / len(dataloader_dev))
    metrics['accuracy']['dev'].append(correct / total)

  return metrics

In [ ]:
metrics = train_loop(model, n_epochs=5, dataloader_train=dl_train, dataloader_dev=dl_dev)
metrics

<br><br><br><br><br><br>

# Part 3 - Evaluation


## Step 1: Evaluation Function

Write an evaluation loop for a trained model using the dev and test datasets. This function will print the `Recall`, `Precision`, and `F1` scores and plot a `Confusion Matrix`.

Perform this evaluation twice:
1. For all labels (7 labels in total).
2. For all labels except "O" (6 labels in total).

## Metrics and Display

### Metrics
- **Recall**: The fraction of true positives that were correctly predicted — TP / (TP + FN). Also known as the True Positive Rate (TPR).
- **Precision**: The fraction of predicted positives that are actually correct — TP / (TP + FP).
- **F1 Score**: The harmonic mean of Precision and Recall.

*Note*: For all these metrics, use **weighted** averaging:
Calculate metrics for each label, and find their average weighted by support. Refer to the [sklearn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html#sklearn.metrics.precision_recall_fscore_support) for more details.

### Display
1. Print the `Recall`, `Precision`, and `F1` scores in a tabulated format.
2. Display a `Confusion Matrix` plot:
   - Rows represent the predicted labels.
   - Columns represent the true labels.
   - Include a title for the plot, axis names, and the names of the tags on the X-axis.

In [ ]:
def evaluate(model: NERNet, title: str, dataloader: DataLoader, vocab: Vocab):
  model.eval()
  all_preds, all_labels = [], []

  with th.no_grad():
    for word_ids, tag_ids in dataloader:
      word_ids, tag_ids = word_ids.to(DEVICE), tag_ids.to(DEVICE)
      output = model(word_ids)
      preds = output.argmax(dim=-1)
      mask = tag_ids != -100
      all_preds.extend(preds[mask].cpu().tolist())
      all_labels.extend(tag_ids[mask].cpu().tolist())

  tag_names = [vocab.id2tag[i] for i in range(vocab.n_tags)]

  # All labels
  p, r, f, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
  print(tabulate([['All labels', r, p, f]], headers=['', 'Recall', 'Precision', 'F1'], tablefmt='psql', floatfmt='.4f'))

  # Without O
  o_id = vocab.tag2id['O']
  filtered = [(l, q) for l, q in zip(all_labels, all_preds) if l != o_id]
  labels_wo_o, preds_wo_o = zip(*filtered) if filtered else ([], [])
  p_wo, r_wo, f_wo, _ = precision_recall_fscore_support(labels_wo_o, preds_wo_o, average='weighted', zero_division=0)
  print(tabulate([['Without O', r_wo, p_wo, f_wo]], headers=['', 'Recall', 'Precision', 'F1'], tablefmt='psql', floatfmt='.4f'))

  # Confusion matrix — rows=predicted, cols=true (as required)
  cm = confusion_matrix(all_labels, all_preds, labels=list(range(vocab.n_tags))).T
  fig, ax = plt.subplots(figsize=(10, 8))
  sns.heatmap(cm, annot=True, fmt='d', ax=ax, xticklabels=tag_names, yticklabels=tag_names)
  ax.set_title(title)
  ax.set_xlabel('True label')
  ax.set_ylabel('Predicted label')
  plt.tight_layout()
  plt.show()

  return {
    'RECALL': r, 'PRECISION': p, 'F1': f,
    'RECALL_WO_O': r_wo, 'PRECISION_WO_O': p_wo, 'F1_WO_O': f_wo
  }

## Step 2: Train & Evaluate on Dev Set

Train and evaluate (on the dev set) a few models, all with `embedding_size=300` and `N_EPOCHS=5` (for fairness and computational reasons), and with the following hyper parameters (you may use that as captions for the models as well):

- Model 1: (hidden_size: 500, n_layers: 1, directions: 1)
- Model 2: (hidden_size: 500, n_layers: 2, directions: 1)
- Model 3: (hidden_size: 500, n_layers: 3, directions: 1)
- Model 4: (hidden_size: 500, n_layers: 1, directions: 2)
- Model 5: (hidden_size: 500, n_layers: 2, directions: 2)
- Model 6: (hidden_size: 500, n_layers: 3, directions: 2)
- Model 7: (hidden_size: 800, n_layers: 1, directions: 2)
- Model 8: (hidden_size: 800, n_layers: 2, directions: 2)
- Model 9: (hidden_size: 800, n_layers: 3, directions: 2)




In [ ]:
N_EPOCHS = 5
EMB_DIM = 300

Here is an example (random numbers) of the display of the results):

In [ ]:
# Example:
results_acc = np.random.rand(9, 10)
columns = ['N_MODEL','HIDDEN_SIZE','N_LAYERS','DIRECTIONS','RECALL','PRECISION','F1','RECALL_WO_O','PRECISION_WO_O','F1_WO_O']
df = pd.DataFrame(results_acc, columns=columns)
df.N_MODEL = [f'model_{n}' for n in range(1,10)]
print(tabulate(df, headers='keys', tablefmt='psql',floatfmt=".4f"))

In [ ]:
N_EPOCHS = 5
EMB_DIM = 300

# Define models with their hyperparameters
models = {
  'Model1': {'embedding_size': EMB_DIM, 'hidden_size': 500, 'n_layers': 1, 'directions': 1},
  'Model2': {'embedding_size': EMB_DIM, 'hidden_size': 500, 'n_layers': 2, 'directions': 1},
  'Model3': {'embedding_size': EMB_DIM, 'hidden_size': 500, 'n_layers': 3, 'directions': 1},
  'Model4': {'embedding_size': EMB_DIM, 'hidden_size': 500, 'n_layers': 1, 'directions': 2},
  'Model5': {'embedding_size': EMB_DIM, 'hidden_size': 500, 'n_layers': 2, 'directions': 2},
  'Model6': {'embedding_size': EMB_DIM, 'hidden_size': 500, 'n_layers': 3, 'directions': 2},
  'Model7': {'embedding_size': EMB_DIM, 'hidden_size': 800, 'n_layers': 1, 'directions': 2},
  'Model8': {'embedding_size': EMB_DIM, 'hidden_size': 800, 'n_layers': 2, 'directions': 2},
  'Model9': {'embedding_size': EMB_DIM, 'hidden_size': 800, 'n_layers': 3, 'directions': 2},
}

columns = ['N_MODEL','HIDDEN_SIZE','N_LAYERS','DIRECTIONS','RECALL','PRECISION','F1','RECALL_WO_O','PRECISION_WO_O','F1_WO_O']
results_dev = pd.DataFrame(columns=columns)
trained_models = {}

for model_name, cfg in models.items():
  m = NERNet(vocab.n_words, embedding_size=cfg['embedding_size'], hidden_size=cfg['hidden_size'],
             output_size=vocab.n_tags, n_layers=cfg['n_layers'], directions=cfg['directions'])
  train_loop(m, n_epochs=N_EPOCHS, dataloader_train=dl_train, dataloader_dev=dl_dev)
  res = evaluate(m, title=model_name, dataloader=dl_dev, vocab=vocab)
  trained_models[model_name] = m
  row = {'N_MODEL': model_name, 'HIDDEN_SIZE': cfg['hidden_size'], 'N_LAYERS': cfg['n_layers'],
         'DIRECTIONS': cfg['directions'], **res}
  results_dev = pd.concat([results_dev, pd.DataFrame([row])], ignore_index=True)

print(tabulate(results_dev, headers='keys', tablefmt='psql', floatfmt=".4f"))

## Step 3: Evaluate on Test Set
Evaluate your models on the test set and save the results as a CSV.

In [ ]:
results = pd.DataFrame(columns=columns)
file_name = "NER_results.csv"

for model_name, m in trained_models.items():
  cfg = models[model_name]
  res = evaluate(m, title=model_name, dataloader=dl_test, vocab=vocab)
  row = {'N_MODEL': model_name, 'HIDDEN_SIZE': cfg['hidden_size'], 'N_LAYERS': cfg['n_layers'],
         'DIRECTIONS': cfg['directions'], **res}
  results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)

results.to_csv(file_name, index=False)
print(tabulate(results, headers='keys', tablefmt='psql', floatfmt=".4f"))

## Step 4 - best model
Decide which model performs the best, write its configuration, train it for **10 epochs total** (5 more than the original 5), and evaluate it on the test set.

In [ ]:
best_model_cfg = {'embedding_size': EMB_DIM, 'hidden_size': -1, 'n_layers': -1, 'directions': -1}

best_idx = results_dev['F1_WO_O'].astype(float).idxmax()
best_row = results_dev.iloc[best_idx]
best_model_cfg['hidden_size']  = int(best_row['HIDDEN_SIZE'])
best_model_cfg['n_layers']     = int(best_row['N_LAYERS'])
best_model_cfg['directions']   = int(best_row['DIRECTIONS'])
print("Best model config:", best_model_cfg)

best_model = NERNet(vocab.n_words, embedding_size=EMB_DIM, hidden_size=best_model_cfg['hidden_size'],
                    output_size=vocab.n_tags, n_layers=best_model_cfg['n_layers'], directions=best_model_cfg['directions'])
train_loop(best_model, n_epochs=10, dataloader_train=dl_train, dataloader_dev=dl_dev)
evaluate(best_model, title="Best Model — 10 epochs", dataloader=dl_test, vocab=vocab)

<br><br><br><br><br>

# Part 4 - Pretrained Embeddings



To prepare for this task, please read [this discussion](https://discuss.pytorch.org/t/can-we-use-pre-trained-word-embeddings-for-weight-initialization-in-nn-embedding/1222).

**TIP**: Ensure that the vectors are aligned with the IDs in your vocabulary. In other words, make sure that the word with ID 0 corresponds to the first vector in the GloVe matrix used to initialize `nn.Embedding`.



## Step 1: Get Data



Download the GloVe embeddings from [this link](https://nlp.stanford.edu/projects/glove/). Use the 300-dimensional vectors from `glove.6B.zip`.



In [ ]:
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip glove.6B.300d.txt
print("GloVe downloaded.")

## Step 2: Inject Embeddings

Then intialize the `nn.Embedding` module in your `NERNet` with these embeddings, so that you can start your training with pre-trained vectors.

In [ ]:
def get_emb_matrix(filepath: str, vocab: Vocab) -> np.ndarray:
  emb_matrix = np.zeros((len(vocab.word2id), 300))
  with open(filepath, 'r', encoding='utf-8') as f:
    for line in f:
      parts = line.strip().split()
      word = parts[0]
      if word in vocab.word2id:
        emb_matrix[vocab.word2id[word]] = np.array(parts[1:], dtype=np.float32)
  return emb_matrix

In [ ]:
def initialize_from_pretrained_emb(model: NERNet, emb_matrix: np.ndarray):
  model.embedding.weight.data.copy_(th.from_numpy(emb_matrix).float())

In [ ]:
# Read embeddings and inject them to a model
emb_file = 'glove.6B.300d.txt'
emb_matrix = get_emb_matrix(emb_file, vocab)
ner_glove = NERNet(input_size=vocab.n_words, embedding_size=EMB_DIM, hidden_size=500, output_size=vocab.n_tags, n_layers=1, directions=1)
initialize_from_pretrained_emb(ner_glove, emb_matrix)

## Step 3: Evaluate on Test Set

Same as the evaluation process before, please display:

1. Print a `RECALL-PRECISION-F1` scores in a tabulate format.
2. Display a `confusion matrix` plot: where the predicted labels are the rows, and the true labels are the columns.

Make sure to use the title for the plot, axis names, and the names of the tags on the X-axis.

The CSV will be submitted automatically along with your results.

In [ ]:
results = pd.DataFrame(columns=columns)
file_name = "NER_results_glove.csv"
glove_models = {}

for model_name, cfg in models.items():
  m = NERNet(vocab.n_words, embedding_size=cfg['embedding_size'], hidden_size=cfg['hidden_size'],
             output_size=vocab.n_tags, n_layers=cfg['n_layers'], directions=cfg['directions'])
  initialize_from_pretrained_emb(m, emb_matrix)
  train_loop(m, n_epochs=N_EPOCHS, dataloader_train=dl_train, dataloader_dev=dl_dev)
  res = evaluate(m, title=model_name, dataloader=dl_test, vocab=vocab)
  glove_models[model_name] = m
  row = {'N_MODEL': model_name, 'HIDDEN_SIZE': cfg['hidden_size'], 'N_LAYERS': cfg['n_layers'],
         'DIRECTIONS': cfg['directions'], **res}
  results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)

results.to_csv(file_name, index=False)
print(tabulate(results, headers='keys', tablefmt='psql', floatfmt=".4f"))

## Step 4 - best model
Decide which model performs the best, write its configuration, train it for **10 epochs total** (5 more than the original 5), and evaluate it on the test set.

In [ ]:
best_model_glove_cfg = {'embedding_size': EMB_DIM, 'hidden_size': -1, 'n_layers': -1, 'directions': -1}

best_idx = results['F1_WO_O'].astype(float).idxmax()
best_row = results.iloc[best_idx]
best_model_glove_cfg['hidden_size']  = int(best_row['HIDDEN_SIZE'])
best_model_glove_cfg['n_layers']     = int(best_row['N_LAYERS'])
best_model_glove_cfg['directions']   = int(best_row['DIRECTIONS'])
print("Best GloVe model config:", best_model_glove_cfg)

best_model_glove = NERNet(vocab.n_words, embedding_size=EMB_DIM,
                          hidden_size=best_model_glove_cfg['hidden_size'],
                          output_size=vocab.n_tags, n_layers=best_model_glove_cfg['n_layers'],
                          directions=best_model_glove_cfg['directions'])
initialize_from_pretrained_emb(best_model_glove, emb_matrix)
train_loop(best_model_glove, n_epochs=10, dataloader_train=dl_train, dataloader_dev=dl_dev)
evaluate(best_model_glove, title="Best GloVe Model — 10 epochs", dataloader=dl_test, vocab=vocab)

# Part 5 - Error Analysis

In this part, you'll analyze the errors made by your best model to understand its strengths and weaknesses.

## Step 1: Extract Predictions

First, let's extract predictions from your best model on the test set:

In [ ]:
def get_predictions(model, dataloader, vocab, PAD_TOKEN, DEVICE):
    """
    Get predictions from the model on a dataloader.

    Returns:
        - true_tags_list: List of lists of true tag strings
        - pred_tags_list: List of lists of predicted tag strings
        - words_list: List of lists of words
    """
    import torch

    model.eval()
    true_tags_list = []
    pred_tags_list = []
    words_list = []

    with torch.no_grad():
        for batch in dataloader:
            # Dataloader yields (input_ids, labels) pairs.
            input_ids, labels = batch
            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            # Get model predictions.
            outputs = model(input_ids)
            _, predicted = torch.max(outputs, 2)

            # Process each sequence in the batch.
            for i in range(input_ids.size(0)):
                # Get sequence length (ignoring padding).
                seq_len = (input_ids[i] != PAD_TOKEN).sum().item()

                # Convert ids back to tag strings and words.
                true_tags = [vocab.id2tag[tag.item()] for tag in labels[i][:seq_len]]
                pred_tags = [vocab.id2tag[tag.item()] for tag in predicted[i][:seq_len]]
                words = [vocab.id2word[word.item()] for word in input_ids[i][:seq_len]]

                true_tags_list.append(true_tags)
                pred_tags_list.append(pred_tags)
                words_list.append(words)

    return true_tags_list, pred_tags_list, words_list

## Step 2: Helper Functions

Before writing the error analysis, implement two utility functions that you will need.

### 2a: `get_entities_simple(tags)`

**Goal:** Extract named entities from a single IOB tag sequence.

**Input:** A list of IOB tag strings, e.g. `['O', 'B-PER', 'I-PER', 'O', 'B-LOC', 'O']`

**Output:** A list of tuples `(start_idx, end_idx, entity_type)` where:
- `start_idx` is the index of the `B-` tag
- `end_idx` is the index of the last `I-` tag of that entity (or same as `start_idx` if the entity is a single token)
- `entity_type` is the string after `B-`/`I-` (e.g. `"PER"`, `"LOC"`, `"ORG"`)

**Logic:** Walk through the tag list. When you see a `B-X` tag, start a new entity. Keep extending it while the next tags are `I-X` (same type). When the entity ends, append the tuple. Ignore `O` tags.

**Example:**
```python
tags = ['O', 'B-PER', 'I-PER', 'O', 'B-LOC', 'O']
get_entities_simple(tags)
# Returns: [(1, 2, 'PER'), (4, 4, 'LOC')]
```

### 2b: `has_overlap(start1, end1, start2, end2)`

**Goal:** Check if two index spans share at least one position.

**Input:** Four integers — start and end indices of two spans.

**Output:** `True` if they overlap, `False` otherwise.

**Example:**
```python
has_overlap(1, 3, 2, 5)  # True  (overlap at indices 2, 3)
has_overlap(1, 2, 4, 5)  # False (no shared indices)
```

In [ ]:
def get_entities_simple(tags):
    entities = []
    i = 0
    while i < len(tags):
        if tags[i].startswith('B-'):
            etype = tags[i][2:]
            start = i
            end = i
            while end + 1 < len(tags) and tags[end + 1] == f'I-{etype}':
                end += 1
            entities.append((start, end, etype))
            i = end + 1
        else:
            i += 1
    return entities

def has_overlap(start1, end1, start2, end2):
    return start1 <= end2 and start2 <= end1

## Step 3: Implement Entity-Level Error Analysis

**Goal:** Compare the true and predicted entity spans (extracted using `get_entities_simple`) to count and categorize errors at the **entity level**.

**Function signature:**
```python
def simple_analyze_errors(true_tags, pred_tags, words) -> dict
```

**Inputs:**
- `true_tags`: list of lists of tag strings (one inner list per sentence)
- `pred_tags`: list of lists of tag strings (one inner list per sentence)
- `words`: list of lists of word strings (one inner list per sentence)

**What to do — for each sentence:**
1. Extract true entities using `get_entities_simple(true_tags[i])`
2. Extract predicted entities using `get_entities_simple(pred_tags[i])`
3. Classify each **true entity** into one of these categories:
   - **Correct:** A predicted entity has the exact same `(start, end, type)`.
   - **Type error:** A predicted entity has the same `(start, end)` but a **different** type (e.g., true=`PER`, predicted=`ORG`).
   - **Boundary error:** A predicted entity **overlaps** (use `has_overlap`) and has the **same type**, but the start/end indices differ.
   - **Missed:** No predicted entity overlaps with this true entity at all.
4. Also count **Spurious** predictions: predicted entities that do **not** overlap with any true entity.

**Return value — a dictionary:**
```python
{
    'total_entities': int,       # total number of true entities across all sentences
    'correct_entities': int,     # number of exact matches
    'accuracy': float,           # correct_entities / total_entities
    'error_counts': {
        'type_error': int,
        'boundary_error': int,
        'missed': int,
        'spurious': int
    },
    'error_examples': {          # store up to 3 examples per category
        'type_error': [(words, true_entity, pred_entity), ...],
        'boundary_error': [(words, true_entity, pred_entity), ...],
        'missed': [(words, true_entity), ...],
        'spurious': [(words, pred_entity), ...]
    }
}
```

**Example walkthrough:**
```python
true_tags = [['O', 'B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC', 'O']]
pred_tags = [['O', 'B-PER', 'O',     'O', 'B-ORG', 'I-ORG', 'O']]
words     = [['The', 'John', 'Smith', 'visited', 'New', 'York', 'yesterday']]

# True entities:  [(1,2,'PER'), (4,5,'LOC')]
# Pred entities:  [(1,1,'PER'), (4,5,'ORG')]
#
# (1,2,'PER') vs (1,1,'PER') → boundary_error (same type PER, overlapping spans, but end differs)
# (4,5,'LOC') vs (4,5,'ORG') → type_error (same span, different type)
```

In [ ]:
def simple_analyze_errors(true_tags, pred_tags, words):
    total_entities = 0
    correct_entities = 0
    error_counts = {'type_error': 0, 'boundary_error': 0, 'missed': 0, 'spurious': 0,
                    'by_entity_type': defaultdict(int)}
    error_examples = {'type_error': [], 'boundary_error': [], 'missed': [], 'spurious': []}

    for i in range(len(true_tags)):
        true_ents = get_entities_simple(true_tags[i])
        pred_ents = get_entities_simple(pred_tags[i])
        w = words[i]
        total_entities += len(true_ents)

        for ts, te, tt in true_ents:
            matched = False
            for ps, pe, pt in pred_ents:
                if ts == ps and te == pe and tt == pt:
                    correct_entities += 1
                    matched = True
                    break
                elif ts == ps and te == pe and tt != pt:
                    error_counts['type_error'] += 1
                    error_counts['by_entity_type'][tt] += 1
                    if len(error_examples['type_error']) < 3:
                        error_examples['type_error'].append((w, (ts, te, tt), (ps, pe, pt)))
                    matched = True
                    break
                elif has_overlap(ts, te, ps, pe) and tt == pt:
                    error_counts['boundary_error'] += 1
                    error_counts['by_entity_type'][tt] += 1
                    if len(error_examples['boundary_error']) < 3:
                        error_examples['boundary_error'].append((w, (ts, te, tt), (ps, pe, pt)))
                    matched = True
                    break
            if not matched:
                error_counts['missed'] += 1
                error_counts['by_entity_type'][tt] += 1
                if len(error_examples['missed']) < 3:
                    error_examples['missed'].append((w, (ts, te, tt)))

        for ps, pe, pt in pred_ents:
            if not any(has_overlap(ts, te, ps, pe) for ts, te, _ in true_ents):
                error_counts['spurious'] += 1
                if len(error_examples['spurious']) < 3:
                    error_examples['spurious'].append((w, (ps, pe, pt)))

    accuracy = correct_entities / total_entities if total_entities > 0 else 0.0
    return {
        'total_entities': total_entities,
        'correct_entities': correct_entities,
        'accuracy': accuracy,
        'error_counts': error_counts,
        'error_examples': error_examples
    }

## Step 4: Display the Error Analysis

**Goal:** Print a readable summary of the error analysis dictionary returned by `simple_analyze_errors`.

**Function signature:**
```python
def print_error_analysis(analysis):
```

**Input:** The dictionary returned by `simple_analyze_errors`.

**What to print — three sections:**

**Section 1 — Overall statistics** (use `tabulate` or simple print statements):
```
Total entities:     500
Correct:            420 (84.0%)
Type errors:         15
Boundary errors:     25
Missed:              40
Spurious:            30
```

**Section 2 — Examples for each error type.** Print up to 3 examples per category, showing the sentence words and the relevant entity span. Format each example like:
```
[Type Error] "... New York ..." — true: LOC(4,5), predicted: ORG(4,5)
[Missed]     "... John Smith ..." — true: PER(1,2), predicted: (none)
```

**Section 3 — Which entity type (PER / LOC / ORG) has the most errors?** Print one line stating the answer.

In [ ]:
def print_error_analysis(analysis):
    total   = analysis['total_entities']
    correct = analysis['correct_entities']
    acc     = analysis['accuracy']
    ec      = analysis['error_counts']
    ee      = analysis['error_examples']

    # Section 1: Overall stats
    print(tabulate([
        ['Total entities',  total],
        ['Correct',         f"{correct} ({acc*100:.1f}%)"],
        ['Type errors',     ec['type_error']],
        ['Boundary errors', ec['boundary_error']],
        ['Missed',          ec['missed']],
        ['Spurious',        ec['spurious']],
    ], tablefmt='simple'))

    # Section 2: Examples per error type
    print()
    for w, true_ent, pred_ent in ee['type_error']:
        ts, te, tt = true_ent; ps, pe, pt = pred_ent
        snippet = ' '.join(w[max(0, ts-1):te+2])
        print(f"[Type Error]     \"...{snippet}...\" — true: {tt}({ts},{te}), predicted: {pt}({ps},{pe})")
    for w, true_ent, pred_ent in ee['boundary_error']:
        ts, te, tt = true_ent; ps, pe, pt = pred_ent
        snippet = ' '.join(w[max(0, ts-1):te+2])
        print(f"[Boundary Error] \"...{snippet}...\" — true: {tt}({ts},{te}), predicted: {pt}({ps},{pe})")
    for w, true_ent in ee['missed']:
        ts, te, tt = true_ent
        snippet = ' '.join(w[max(0, ts-1):te+2])
        print(f"[Missed]         \"...{snippet}...\" — true: {tt}({ts},{te}), predicted: (none)")
    for w, pred_ent in ee['spurious']:
        ps, pe, pt = pred_ent
        snippet = ' '.join(w[max(0, ps-1):pe+2])
        print(f"[Spurious]       \"...{snippet}...\" — predicted: {pt}({ps},{pe}), true: (none)")

    # Section 3: Worst entity type
    by_type = ec['by_entity_type']
    if by_type:
        worst = max(by_type, key=by_type.get)
        print(f"\nEntity type with most errors: {worst} ({by_type[worst]} errors)")

## Step 5: Improvement Suggestions

**Goal:** Based on the output of your error analysis, write **3 concrete improvement suggestions** below (no code required — just text).

**Requirements:**
- Write 2–4 sentences per suggestion.
- Each suggestion **must reference a specific finding** from your error analysis (e.g., "42% of errors are boundary errors on PER entities, which suggests...").
- Choose 3 of the following directions:
  1. Using a **CRF layer** on top of the LSTM to enforce valid IOB transitions
  2. Using **subword or character-level embeddings** to handle rare / OOV entity words
  3. Adding more **training data or data augmentation** for the weakest entity type
  4. Using **contextual embeddings** (e.g., BERT) instead of static word embeddings
  5. **Post-processing rules** to fix common boundary errors

**Expected output:** A markdown cell (below) with three numbered paragraphs.

In [ ]:
# First, extract predictions from your best model on the test set
true_tags_list, pred_tags_list, words_list = get_predictions(model, dl_test, vocab, PAD_TOKEN, DEVICE)

# Run the error analysis
analysis = simple_analyze_errors(true_tags_list, pred_tags_list, words_list)

# Display the results
print_error_analysis(analysis)

### Your Improvement Suggestions

*(Write your 3 suggestions here)*

1. ...

2. ...

3. ...

# Testing

Before running the tests:
1. Create a **sharing link** to your notebook with **editor access**.
2. Paste it in the `NOTEBOOK_LINK` variable below.

Then run the test cells to create the `results.json` file.

In [ ]:
NOTEBOOK_LINK = None

In [ ]:
########################################
# Tests

import json

def test_link():
    return {
        'link': NOTEBOOK_LINK
    }

train_ds = read_data("data/train.txt")
dev_ds = read_data("data/dev.txt")
test_ds = read_data("data/test.txt")
def test_read_data():
    result = {
        'lengths': (len(train_ds), len(dev_ds), len(test_ds)),
    }
    return result

vocab = Vocab(train_ds)
def test_vocab():
    sent = vocab.index_words(["I", "am", "Spongebob"])
    return {
        'length': vocab.n_words,
        'tag2id_length': len(vocab.tag2id),
        "Spongebob": sent[2]
    }

train_sequences = prepare_data(train_ds, vocab)
dev_sequences = prepare_data(dev_ds, vocab)
test_sequences = prepare_data(test_ds, vocab)

def test_count_oov():
    return {
        'dev_oov': count_oov(dev_sequences),
        'test_oov': count_oov(test_sequences)
    }

BATCH_SIZE = 16
dl_train = prepare_data_loader(train_sequences, batch_size=BATCH_SIZE)
dl_dev = prepare_data_loader(dev_sequences, batch_size=BATCH_SIZE, train=False)
dl_test = prepare_data_loader(test_sequences, batch_size=BATCH_SIZE, train=False)

def test_prepare_data_loader():
    return {
        'lengths': (len(dl_train), len(dl_dev), len(dl_test))
    }


def test_NERNet():
    # Extract best model configuration
    hidden_size = best_model_cfg['hidden_size']
    n_layers = best_model_cfg['n_layers']
    directions = best_model_cfg['directions']


    # Create model
    best_model = NERNet(vocab.n_words, embedding_size=300, hidden_size=hidden_size, output_size=vocab.n_tags, n_layers=n_layers, directions=directions)
    best_model.to(DEVICE)

    # Train model and evaluate
    _ = train_loop(best_model, n_epochs=10, dataloader_train=dl_train, dataloader_dev=dl_dev)
    results = evaluate(best_model, title="", dataloader=dl_test, vocab=vocab)

    return {
        'f1': results['F1'],
        'f1_wo_o': results['F1_WO_O'],
    }

def test_glove():
    # Get embeddings
    emb_file = 'glove.6B.300d.txt'
    emb_matrix = get_emb_matrix(emb_file, vocab)

    # Extract best model configuration
    hidden_size = best_model_glove_cfg['hidden_size']
    n_layers = best_model_glove_cfg['n_layers']
    directions = best_model_glove_cfg['directions']

    # Create model
    best_model = NERNet(vocab.n_words, embedding_size=300, hidden_size=hidden_size, output_size=vocab.n_tags, n_layers=n_layers, directions=directions)
    best_model.to(DEVICE)
    initialize_from_pretrained_emb(best_model, emb_matrix)

    # Train model and evaluate
    _ = train_loop(best_model, n_epochs=10, dataloader_train=dl_train, dataloader_dev=dl_dev)
    results = evaluate(best_model, title="", dataloader=dl_test, vocab=vocab)

    return {
        'f1': results['F1'],
        'f1_wo_o': results['F1_WO_O'],
    }

TESTS = [
    test_link,
    test_read_data,
    test_vocab,
    test_count_oov,
    test_prepare_data_loader,
    test_NERNet,
    test_glove
]

# Run tests and save results
res = {}
for test in TESTS:
    try:
        cur_res = test()
        res.update({test.__name__: cur_res})
    except Exception as e:
        import traceback
        res.update({test.__name__: repr(e) + "\n" + traceback.format_exc()})

with open('results.json', 'w') as f:
    json.dump(res, f, indent=2)

########################################

---

# 📤 Submit Your Assignment to GitHub

## Step 1: Authentication Setup (One-Time)

Before you can submit, you need to set up GitHub authentication.

### Creating a GitHub Personal Access Token:

1. **Go to GitHub Token Settings**: [https://github.com/settings/tokens](https://github.com/settings/tokens)

2. **Click "Generate new token (classic)"**

3. **Configure your token**:
   - **Note**: "NLP Course Assignments" (or any name you like)
   - **Expiration**: 90 days (or custom)
   - **Select scopes**: Check **`repo`** (full control of private repositories)

4. **Click "Generate token"**

5. **IMPORTANT**: Copy the token immediately and save it somewhere safe!
   - Like Colab Secrets (see picture)
   - You won't be able to see it again
   - You can reuse this token for all assignments
   - Don't share it with anyone

### Run the authentication cell below

You only need to do this **once per Colab session**. If you restart the runtime, you'll need to re-run the authentication cell.

---

In [ ]:
"""
GitHub Authentication Setup
Run this cell ONCE to set up your GitHub credentials
"""

import os
from getpass import getpass

def setup_github_auth():
    """Set up GitHub credentials - run once per Colab session"""
    global GITHUB_USERNAME, GITHUB_TOKEN

    print("🔐 GitHub Authentication Setup")
    print("=" * 60)

    GITHUB_USERNAME = input("GitHub username: ")
    GITHUB_TOKEN = getpass("GitHub Personal Access Token (hidden): ")

    print("\n✅ Credentials saved for this session!")
    print("You can now run the submission cell below.")
    print("\n💡 Tip: Your credentials are only stored in this runtime.")
    print("If you restart the runtime, you'll need to run this cell again.")

# Run the setup
setup_github_auth()

---

## Step 2: Submit Your Results

Once you've:
- ✅ Completed all the code cells above
- ✅ Run all the test cells
- ✅ Generated `results.json`
- ✅ Run the authentication cell

You can now submit your assignment by running the cell below!

### What you'll need:
- Your **GitHub Classroom repository URL**
  - You received this when you accepted the assignment
  - Format: `https://github.com/NLP-Reichman/2026-assignment-2-team-name`
- (Optional) A custom commit message

### After submission:
- Check your repository to see `results.json` has been uploaded
- Visit the **Actions** tab to see your autograding results
- Results typically appear within 1-2 minutes

---

In [ ]:
"""
Submit Assignment to GitHub
Run this cell to push your results.json to GitHub
"""

import os
import subprocess
import json

def check_credentials():
    """Check if credentials are set"""
    try:
        _ = GITHUB_USERNAME
        _ = GITHUB_TOKEN
        return True
    except NameError:
        print("\u274c GitHub credentials not found!")
        print("Please run the authentication cell above first.")
        return False


def check_results_file():
    """Check if results.json exists"""
    if not os.path.exists('results.json'):
        print("\u274c results.json not found!")
        print("\nPlease run all the test cells above to generate results.json")
        return False

    # Display test summary
    try:
        with open('results.json', 'r') as f:
            results = json.load(f)

        print("\U0001f4ca Test Results Found:")
        print("-" * 60)
        for test_name in results.keys():
            print(f"  \u2713 {test_name}")
        print("-" * 60)
        return True
    except Exception as e:
        print(f"\u26a0\ufe0f  Warning: Could not read results.json: {e}")
        return True  # Still allow submission


def submit_to_github(repo_url, commit_message=None):
    """Submit results.json to GitHub repository"""

    if commit_message is None:
        commit_message = "Submit assignment results from Colab"

    print("\n\U0001f680 Submitting to GitHub...")
    print("=" * 60)

    # Create temporary directory
    temp_dir = '/content/github_submission'
    if os.path.exists(temp_dir):
        subprocess.run(['rm', '-rf', temp_dir], check=True, capture_output=True)

    os.makedirs(temp_dir, exist_ok=True)
    os.chdir(temp_dir)

    try:
        # Configure git
        subprocess.run(['git', 'config', '--global', 'user.email',
                       f'{GITHUB_USERNAME}@users.noreply.github.com'],
                      check=True, capture_output=True)
        subprocess.run(['git', 'config', '--global', 'user.name',
                       GITHUB_USERNAME],
                      check=True, capture_output=True)

        # Clone repository with authentication
        auth_url = repo_url.replace('https://', f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@')

        print("\U0001f4e5 Cloning repository...")
        result = subprocess.run(['git', 'clone', auth_url, 'repo'],
                              capture_output=True, text=True)

        if result.returncode != 0:
            print(f"\u274c Error cloning repository:")
            print(result.stderr)
            print("\n\U0001f4a1 Troubleshooting:")
            print("  - Check that your repository URL is correct")
            print("  - Verify your token has 'repo' scope")
            print("  - Make sure you've accepted the assignment")
            return False

        # Change to repo directory
        os.chdir('repo')

        # Copy results.json
        print("\U0001f4dd Copying results.json...")
        subprocess.run(['cp', '/content/results.json', 'results.json'],
                      check=True, capture_output=True)

        # Copy CSV result files if they exist
        for csv_file in ['NER_results.csv', 'NER_results_glove.csv']:
            csv_path = f'/content/{csv_file}'
            if os.path.exists(csv_path):
                print(f"\U0001f4dd Copying {csv_file}...")
                subprocess.run(['cp', csv_path, csv_file],
                              check=True, capture_output=True)

        # Check for changes
        status = subprocess.run(['git', 'status', '--porcelain'],
                              capture_output=True, text=True)

        if not status.stdout.strip():
            print("\n\u2139\ufe0f  No changes detected - results.json is unchanged")
            print("\u2705 Your repository is already up to date!")
            return True

        # Commit and push
        print(f"\U0001f4ac Commit message: '{commit_message}'")
        print("\U0001f4e4 Pushing to GitHub...")

        subprocess.run(['git', 'add', 'results.json',
                        'NER_results.csv', 'NER_results_glove.csv'],
                      check=True, capture_output=True)
        subprocess.run(['git', 'commit', '-m', commit_message],
                      check=True, capture_output=True)
        subprocess.run(['git', 'push'],
                      check=True, capture_output=True)

        print("\n" + "=" * 60)
        print("\u2705 SUCCESS! Assignment submitted!")
        print("=" * 60)
        print(f"\n\U0001f4ca Repository: {repo_url}")
        print(f"\U0001f4ca Autograding: {repo_url.replace('.git', '')}/actions")
        print("\n\U0001f4a1 Your grade will appear in the Actions tab in ~1 minute")

        return True

    except subprocess.CalledProcessError as e:
        print(f"\n\u274c Git error occurred")
        if hasattr(e, 'stderr') and e.stderr:
            print(f"Details: {e.stderr}")
        return False
    except Exception as e:
        print(f"\n\u274c Unexpected error: {e}")
        return False
    finally:
        # Return to /content
        os.chdir('/content')


def main():
    """Main submission workflow"""
    print("=" * 60)
    print("\U0001f4e4 Assignment Submission")
    print("=" * 60)

    # Check credentials
    if not check_credentials():
        return

    # Check results file
    if not check_results_file():
        return

    # Get repository URL
    print("\n\U0001f4cd Enter your GitHub Classroom repository URL")
    print("Example: https://github.com/NLP-Reichman/2026-assignment-2-username")
    repo_url = input("\nRepository URL: ").strip()

    # Validate URL
    if not repo_url.startswith('https://github.com/'):
        print("\u274c Invalid URL - must start with https://github.com/")
        return

    # Get commit message (optional)
    print("\n\U0001f4ac Commit Message (optional)")
    print("Press Enter for default message, or type your own:")
    commit_msg = input("Message: ").strip()

    if not commit_msg:
        commit_msg = "Submit assignment results from Colab"

    # Confirm submission
    print("\n" + "=" * 60)
    print("Ready to submit:")
    print(f"  Repository: {repo_url}")
    print(f"  File: results.json")
    print(f"  Message: {commit_msg}")
    print("=" * 60)

    confirm = input("\nProceed? (yes/no): ").strip().lower()

    if confirm in ['yes', 'y']:
        success = submit_to_github(repo_url, commit_msg)
        if success:
            print("\n\U0001f389 All done!")
    else:
        print("\n\u274c Submission cancelled")


# Run submission
main()